In [ ]:
from openai import AzureOpenAI, OpenAI
import tiktoken

from markitdown import MarkItDown
from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

import psycopg2
from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector

import numpy as np
import pdfplumber
from sentence_transformers import SentenceTransformer

from dotenv import load_dotenv
import os
import re

from utils.chunking import *
from utils.embedding import *

load_dotenv("project.env")

In [ ]:
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


In [31]:
path = "./data/"
fname = "euaiact.pdf"
pdf_path = os.path.join(path, fname)

In [36]:
# read the pdf and extract text
pdf_text = extract_text_from_pdf(pdf_path)

In [32]:
res = chunk_semantically_spacy(pdf_text)

In [42]:
# count tokens in the first chunk
tokenizer = tiktoken.get_encoding("cl100k_base")
print(len(res), len(tokenizer.encode(res[3])))

503 175


In [53]:
names = [
    'Qwen/Qwen3-Embedding-0.6B',
    'Qwen/Qwen3-Embedding-4B',
    'sentence-transformers/all-MiniLM-L6-v2'
]

In [55]:
wrapper = SentenceTransformerWrapper(names[0], device="cpu")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
embeddings = wrapper.embed_documents(res)

In [ ]:
conn_string = "postgresql://{user}:{password}@{host}:{port}/{database}".format(
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST"),
    port=5431,
    database=os.getenv("PGDATABASE")
)

In [ ]:
try:
    conn = psycopg2.connect(conn_string)
    print("Connection successful")
    with conn.cursor() as cur:
        register_vector(cur)
        cur.execute("SELECT 1")
        print(cur.fetchone())
except Exception as e:
    print("Connection failed")
    print(e)


In [ ]:
with conn.cursor() as cur:
    for chunk in res:
        emb = wrapper.embed_query(chunk)
        cur.execute("INSERT INTO documents (content, embedding) VALUES (%s, %s)", (chunk, emb))
        # or if cast is necesary for embeddings:
        # cur.execute("INSERT INTO documents (content, embedding) VALUES (%s, %s::vector)", (chunk, emb))
    conn.commit()
